# Vector Stores

Stores text **and** vectors side-by-side so you can retrieve chunks by semantic similarity at query time.

## The Pipeline So Far

`ingest -> split -> embed -> **store** -> retrieve -> answer`

A vector store holds two things per chunk:
- `page_content` (the original text)
- `embedding` (the vector)

Plus the chunk's `metadata`, which you can filter on.

## The LangChain VectorStore Interface

- `from_documents(docs, embedding)` -- bulk ingest, returns a VectorStore
- `similarity_search(query, k=4)` -- returns the `k` most similar `Document`s
- `similarity_search_with_score(query, k)` -- same but also returns distance
- `save_local(path)` / `load_local(path)` -- persist to disk (provider-dependent)

Providers: FAISS, Chroma, Pinecone, Qdrant, Weaviate, etc.

---
## Setup: load + split + embed

All examples below reuse the same chunks from `data_ingestion/`.

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# --- find the data_ingestion dir from wherever we are ---
def find_data_dir():
    for root in (Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]):
        cand = root / "data_ingestion"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("Could not find data_ingestion folder")

DATA = find_data_dir()
print("Using data dir:", DATA)

# --- load both files ---
lyrics = TextLoader(str(DATA / "heylog_12_gauge_lyrics.txt")).load()
para   = TextLoader(str(DATA / "random_paragraph.txt")).load()

# add a "category" field so we can filter by source type later
for d in lyrics:
    d.metadata["category"] = "lyrics"
for d in para:
    d.metadata["category"] = "paragraph"

# --- split into chunks ---
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks   = splitter.split_documents(lyrics + para)

# --- embeddings (local Ollama) ---
emb = OllamaEmbeddings(model="nomic-embed-text")  # 768 dims, local

print(f"{len(chunks)} chunks ready")
print("sample metadata:", chunks[0].metadata)


/tmp/ipykernel_298815/3695690387.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/home/vanitas/Python/git/elusive_Agentic_AI/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using data dir: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion
40 chunks ready
sample metadata: {'source': '/home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt', 'category': 'lyrics'}


---
## FAISS (Facebook AI Similarity Search)

- **In-memory** -- fast, no server needed
- Saves to disk as a single folder (index + docs pickle)
- No metadata filtering (unlike Chroma) -- just pure similarity search
- Good for: large corpora, prototyping, anything local

In [2]:
from langchain_community.vectorstores import FAISS

# from_documents: bulk ingest chunks + embeddings -> returns a VectorStore
faiss_store = FAISS.from_documents(chunks, emb)
print("FAISS index size:", faiss_store.index.ntotal)


FAISS index size: 40


In [4]:

# similarity_search: returns top-k Document objects
results = faiss_store.similarity_search("dreaming of someone", k=5)

for i, doc in enumerate(results):
    print(f"\n[{i}]")
    print(f"    source: {doc.metadata['source']}")
    print(f"    text:   {doc.page_content[:80]}")



[0]
    source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
    text:   And this what it sound like when I dream of you

[1]
    source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
    text:   [Verse 5]
Shuttin' me down, wake up from REM
I had a dream of us both layin' in 

[2]
    source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
    text:   But when I woke up, I thought that it was real
I tried to go back to sleep to ge

[3]
    source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
    text:   [Verse 4]
You ever think about where I been?
You ever think about who we was?

[4]
    source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
    text:   And I fall asleep, but before I do
I 

In [7]:
# similarity_search_with_score: (Document, distance) tuples
# distance = L2 norm -- lower = more similar
scored = faiss_store.similarity_search_with_score("dreaming of someone", k=10)

for doc, score in scored:
    print(f"distance {score:.4f} | {doc.page_content[:60]}")


distance 0.6725 | And this what it sound like when I dream of you
distance 0.7846 | [Verse 5]
Shuttin' me down, wake up from REM
I had a dream o
distance 0.8451 | But when I woke up, I thought that it was real
I tried to go
distance 0.8495 | [Verse 4]
You ever think about where I been?
You ever think 
distance 0.8647 | And I fall asleep, but before I do
I sit in silence for an h
distance 0.9541 | [Bridge]
Ooh
Ooh-ooh-ooh
Ooh
And she has my attire
And she h
distance 0.9563 | Thoughts begin to overflow
distance 0.9732 | I stare into darkness tryna see objects
The only thing I see
distance 1.0047 | Of what it was like again to see your eyes
I knew it was imp
distance 1.0231 | The librarian, a woman with silver-rimmed glasses and kind e


In [5]:
# similarity_search_with_score: (Document, distance) tuples
# distance = L2 norm -- lower = more similar
scored = faiss_store.similarity_search_with_score("library and silence", k=3)

for doc, score in scored:
    print(f"distance {score:.4f} | {doc.page_content[:60]}")


distance 0.4863 | and the man and the library grew older together, content in 
distance 0.6383 | The old library smelled of paper dust and warm wood, and eve
distance 0.8040 | The librarian, a woman with silver-rimmed glasses and kind e


In [8]:

# save/load: persist index to disk so you don't re-embed next time
faiss_store.save_local("faiss_index")
print("saved faiss_index/")

reloaded = FAISS.load_local(
    "faiss_index",
    emb,
    allow_dangerous_deserialization=True,  # needed for pickle format
)
print("reloaded, index size:", reloaded.index.ntotal)


saved faiss_index/
reloaded, index size: 40


---
## Chroma

- **In-memory by default**, can persist to a local directory
- Supports **metadata filtering** at query time
- Stores its own database file on disk
- Good for: prototyping + smaller production, when you need filters

In [9]:
from langchain_community.vectorstores import Chroma

# persist_directory is created automatically
chroma_store = Chroma.from_documents(
    chunks,
    emb,
    persist_directory="chroma_db",
    collection_name="lyrics",
)
print("Chroma collection size:", chroma_store._collection.count())


Chroma collection size: 40


In [11]:
# similarity search -- same API as FAISS
results = chroma_store.similarity_search("vivid thoughts and silence", k=4)

for doc in results:
    print(f"source: {doc.metadata['source']}")
    print(f"  {doc.page_content[:70]}\n")


source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
  And I fall asleep, but before I do
I sit in silence for an hour or two

source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
  Thoughts begin to overflow

source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/heylog_12_gauge_lyrics.txt
  And this what it sound like when I dream of you

source: /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain/data_ingestion/random_paragraph.txt
  and the man and the library grew older together, content in a silence 



In [12]:

# similarity search with score
scored = chroma_store.similarity_search_with_score("clothes and wardrobe", k=3)
for doc, score in scored:
    print(f"distance {score:.4f} | {doc.page_content[:60]}")


distance 0.6824 | I wonder what is stoppin' me
She loves borrowin' my wardrobe
distance 0.7177 | Said I smell good, so you wear all of my clothes (My clothes
distance 0.7299 | Next thing you know she has my whole attire


In [13]:
# metadata filtering: only search chunks where category == "lyrics"
filtered_store = Chroma.from_documents(
    chunks,
    emb,
    persist_directory="chroma_filtered",
    collection_name="lyrics_filtered",
)

# filter on the "category" metadata field we added during setup
results = filtered_store.similarity_search(
    "dreams and sleeping",
    k=2,
    filter={"category": "lyrics"},  # skip paragraph chunks
)

print("filtered results:")
for doc in results:
    print(f"  [{doc.metadata['category']}] {doc.page_content[:70]}")


filtered results:
  [lyrics] And I fall asleep, but before I do
I sit in silence for an hour or two
  [lyrics] [Verse 5]
Shuttin' me down, wake up from REM
I had a dream of us both 


---
## FAISS vs Chroma

| | FAISS | Chroma |
|---|---|---|
| Metadata filtering | No | Yes |
| Speed (large corpus) | Faster | Good |
| Persistence | save_local / load_local | persist_directory |
| Default in-memory | Yes | Yes (also can persist) |
| Dependencies | faiss-cpu / faiss-gpu | chromadb |

Both use the same `Embeddings` interface -- swap by changing one line.

---
## Conclusion

**What a vector store gives you:**
- Semantic search over your documents (not just keyword match)
- Metadata to filter before or after searching
- Persistence so you don't re-embed every run

**Next step:** feed the retrieved chunks into an LLM via a RAG chain -- that's where the `RetrievalQA` / `create_retrieval_chain` step comes in.